# Climate Module — Aligned Ida Replay

**Project:** PLEXA  
**Author:** Muhammad Kaleem  
**Role:** Climate Module Lead

## Objective
Prepare climate inputs and actual predictions for the four LION roads
requested by Anas, covering September 1–3, 2021, New York time,
with September 3 excluded.

## Approach
Review road mappings, verify hourly feature coverage, and export
predictions with mapping evidence and source provenance.

## Model Policy
Use the existing model, features, history policies and threshold.
No retraining is performed. Zero alerts are retained.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "config/climate_history_release_v3_02.json").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

REQUEST_DIR = (
    Path.home()
    / "Downloads/climate_aligned_input_request_v0.1.0"
)


def read_json(path):
    return json.loads(path.read_text())


def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


# Verify the files against Anas's supplied manifest.
request_manifest = read_json(REQUEST_DIR / "manifest.json")

for name, expected in request_manifest["files"].items():
    path = REQUEST_DIR / name

    if not path.is_file():
        raise FileNotFoundError(path)

    if (
        path.stat().st_size != expected["bytes"]
        or file_sha256(path) != expected["sha256"]
    ):
        raise ValueError(f"Request file verification failed: {name}")

request_spec = read_json(REQUEST_DIR / "request_spec.json")
requested_geojson = read_json(
    REQUEST_DIR / "requested_lion_roads.geojson"
)

# Check the requested road identifiers.
requested_roads = pd.DataFrame([
    feature["properties"]
    for feature in requested_geojson["features"]
])

actual_ids = requested_roads["lion_segment_id"].astype(str)
expected_ids = request_spec["requested_lion_segments"]

if actual_ids.duplicated().any() or set(actual_ids) != set(expected_ids):
    raise ValueError("Road geometries do not match the requested IDs.")

# Verify 48 consecutive hours using the New York calendar dates.
START_UTC = pd.Timestamp(
    "2021-09-01", tz="America/New_York"
).tz_convert("UTC")
END_UTC = pd.Timestamp(
    "2021-09-03", tz="America/New_York"
).tz_convert("UTC")

expected_hours = pd.date_range(
    START_UTC, END_UTC, freq="h", inclusive="left"
)
requested_hours = pd.DatetimeIndex(pd.to_datetime(
    request_spec["requested_hour_starts_utc"], utc=True
))

if not requested_hours.equals(expected_hours):
    raise ValueError("Requested hours differ from the agreed Ida window.")

# Verify the requested release without loading or changing the model.
release = read_json(
    PROJECT_ROOT / "config/climate_history_release_v3_02.json"
)
BUNDLE_DIR = PROJECT_ROOT / release["bundle_path"]

if release["release_id"] != request_spec["existing_climate_package_id"]:
    raise ValueError("Requested model package differs from our release.")

if file_sha256(BUNDLE_DIR / "release_manifest.json") != (
    request_spec["existing_climate_manifest_sha256"]
):
    raise ValueError("Model release manifest hash does not match.")

if release["threshold"] != request_spec["model_policy"]["alert_threshold"]:
    raise ValueError("Requested threshold differs from our release.")

display(requested_roads)

print("Request files: verified")
print("Model release:", release["release_id"])
print("Start UTC:", START_UTC)
print("End UTC, exclusive:", END_UTC)
print("Requested hours per supported climate segment:", len(expected_hours))
print("Road mapping remains draft pending review.")
print("No files changed, predictions generated or training performed.")

,lion_segment_id,street_name,supported_mobility_directions
0,21148,50 STREET,EB
1,38916,WEST 145 STREET,"EB,WB"
2,81165,BROADWAY,"NB,SB"
3,96010,FRESH MEADOW LANE,"NB,SB"


Request files: verified
Model release: climate-history-v3-02-experimental-01
Start UTC: 2021-09-01 04:00:00+00:00
End UTC, exclusive: 2021-09-03 04:00:00+00:00
Requested hours per supported climate segment: 48
Road mapping remains draft pending review.
No files changed, predictions generated or training performed.


## 2. Proposed Road Mapping

Compare the requested LION geometries with climate physical segments
using distances and buffered overlap in metres.

Retain nearby alternatives for review. Geometry matches are proposals,
not an approved crosswalk.

In [2]:
import geopandas as gpd

METRIC_CRS = "EPSG:32618"
SEARCH_DISTANCE_M = 20
OVERLAP_BUFFER_M = 10

lion_roads = gpd.read_file(
    REQUEST_DIR / "requested_lion_roads.geojson"
)

if lion_roads.crs is None:
    raise ValueError("Requested roads have no coordinate reference system.")

climate_segments = gpd.read_parquet(
    PROJECT_ROOT
    / "data/processed/baseline_street_features_v1/street_segments.parquet"
)

if climate_segments.crs is None:
    raise ValueError("Climate segments have no coordinate reference system.")

if climate_segments["street_segment_id"].duplicated().any():
    raise ValueError("Duplicate climate physical segment IDs.")

lion_metric = lion_roads.to_crs(METRIC_CRS)
climate_metric = climate_segments.to_crs(METRIC_CRS)

for frame in [lion_metric, climate_metric]:
    if (
        frame.geometry.isna().any()
        or frame.geometry.is_empty.any()
        or (~frame.geometry.is_valid).any()
        or (frame.geometry.length <= 0).any()
    ):
        raise ValueError("Missing, invalid or zero-length road geometry.")

candidate_rows = []

for road in lion_metric.itertuples(index=False):
    nearby_indices = climate_metric.sindex.query(
        road.geometry.buffer(SEARCH_DISTANCE_M),
        predicate="intersects",
    )
    nearby = climate_metric.iloc[nearby_indices]

    for segment in nearby.itertuples(index=False):
        lion_length = road.geometry.length
        climate_length = segment.geometry.length

        candidate_rows.append({
            "lion_segment_id": str(road.lion_segment_id),
            "street_name": road.street_name,
            "street_segment_id": segment.street_segment_id,
            "minimum_distance_m": road.geometry.distance(segment.geometry),
            "lion_length_m": lion_length,
            "climate_length_m": climate_length,
            "lion_covered_percent": (
                road.geometry.intersection(
                    segment.geometry.buffer(OVERLAP_BUFFER_M)
                ).length / lion_length * 100
            ),
            "climate_covered_percent": (
                segment.geometry.intersection(
                    road.geometry.buffer(OVERLAP_BUFFER_M)
                ).length / climate_length * 100
            ),
        })

if not candidate_rows:
    raise ValueError("No nearby climate segments found.")

mapping_candidates = pd.DataFrame(candidate_rows).sort_values(
    [
        "lion_segment_id",
        "lion_covered_percent",
        "climate_covered_percent",
        "minimum_distance_m",
        "street_segment_id",
    ],
    ascending=[True, False, False, True, True],
).reset_index(drop=True)

mapping_candidates["candidate_rank"] = (
    mapping_candidates.groupby("lion_segment_id").cumcount() + 1
)

print("Top three geometry candidates per requested road:")
display(
    mapping_candidates.loc[
        mapping_candidates["candidate_rank"] <= 3
    ].style.format({
        "minimum_distance_m": "{:.2f}",
        "lion_length_m": "{:.2f}",
        "climate_length_m": "{:.2f}",
        "lion_covered_percent": "{:.2f}",
        "climate_covered_percent": "{:.2f}",
    })
)

unmatched_ids = sorted(
    set(expected_ids) - set(mapping_candidates["lion_segment_id"])
)

print("Roads without candidates within 20 metres:", unmatched_ids)
print("Overlap percentages use a 10-metre buffer.")
print("All nearby alternatives remain in mapping_candidates.")
print("Ranking is for inspection; no mapping has been accepted.")
print("No scores inspected and no files changed.")

Top three geometry candidates per requested road:


,lion_segment_id,street_name,street_segment_id,minimum_distance_m,lion_length_m,climate_length_m,lion_covered_percent,climate_covered_percent,candidate_rank
0,21148,50 STREET,eec2bba8cf93f627fa05953f4bd4737dd7841ae924a3e94f4779988f2d201ea2,0.00,231.49,231.16,100.00,100.00,1
1,21148,50 STREET,2c62c59d5c00116d36cf6a469ed683c88d6403441b63cbef6b4b45146a05c81c,0.00,231.49,94.33,5.27,12.68,2
2,21148,50 STREET,be3b39832c4a5c287be2fb45351af4a7c2d13dfa5cd539b4ae3cf1baa209d3b0,0.17,231.49,10.29,4.46,97.33,3
12,38916,WEST 145 STREET,43e47dd184082b21bf9c06842044c92803a0744faf884d090a42086f6fbb2efb,0.39,88.05,88.10,100.00,100.00,1
13,38916,WEST 145 STREET,b08862d85d28856e1421129e81d79a55b5a51406133ed27273cd4b378967b525,0.39,88.05,402.04,12.42,2.39,2
14,38916,WEST 145 STREET,39899c72228ca46d17d16256f66935aef6eb6e715f6aff0835fec4d1f4fb8eb7,0.00,88.05,248.17,12.42,4.19,3
19,81165,BROADWAY,c994272aa257dc6b3562635bf8531df0703a3aa45aa93e31f81864fbffb77751,1.50,129.64,130.55,100.00,100.00,1
20,81165,BROADWAY,94dd1b62ce79d7efae700556646bc5d9c109da0f7180952895e757a9a4fe5699,1.50,129.64,133.49,7.53,7.31,2
21,81165,BROADWAY,76c57f356b71514516870a0556983d783021570db631ebc32ba2bc5780766649,1.50,129.64,83.77,7.53,10.15,3
24,96010,FRESH MEADOW LANE,78db30258ddc12939307ca4759d260d5011751c3d26b406a76400d63c0c02824,0.00,169.18,166.36,100.00,100.00,1


Roads without candidates within 20 metres: []
Overlap percentages use a 10-metre buffer.
All nearby alternatives remain in mapping_candidates.
Ranking is for inspection; no mapping has been accepted.
No scores inspected and no files changed.


## 3. Model Input Coverage

Check the proposed climate segments against the saved model bridge,
static street features, and all 48 requested rainfall hours.

Keep unavailable inputs explicit. Road mappings remain draft.

In [3]:
import numpy as np

# Require one strong candidate per LION road.
strong_candidates = mapping_candidates.loc[
    (mapping_candidates["lion_covered_percent"] >= 95)
    & (mapping_candidates["climate_covered_percent"] >= 95)
].copy()

match_counts = (
    strong_candidates.groupby("lion_segment_id")
    .size()
    .reindex(expected_ids, fill_value=0)
)

if not match_counts.eq(1).all():
    raise ValueError(
        "Some roads need manual mapping review:\n"
        + match_counts.to_string()
    )

proposed_mapping = strong_candidates.copy()
proposed_mapping["mapping_status"] = "proposed_pending_review"

static = pd.read_parquet(
    PROJECT_ROOT
    / "data/processed/baseline_street_features_v1/street_features.parquet"
)
bridge = pd.read_parquet(BUNDLE_DIR / "street_cell_bridge.parquet")

weather = pd.read_parquet(
    PROJECT_ROOT
    / "data/processed/baseline_weather_v1/2021-09/weather_features.parquet",
    filters=[
        ("prediction_time_utc", ">=", START_UTC.to_pydatetime()),
        ("prediction_time_utc", "<", END_UTC.to_pydatetime()),
    ],
)

weather["prediction_time_utc"] = pd.to_datetime(
    weather["prediction_time_utc"], utc=True
)

for name, frame in [("static", static), ("bridge", bridge)]:
    if frame["street_segment_id"].duplicated().any():
        raise ValueError(f"Duplicate street IDs in {name}.")

weather_keys = ["grid_id", "cell_id", "prediction_time_utc"]
if weather.duplicated(weather_keys).any():
    raise ValueError("Duplicate rainfall cell-hours.")

# Build all requested segment-hours before checking availability.
expected_rows = proposed_mapping[
    ["lion_segment_id", "street_segment_id"]
].merge(
    pd.DataFrame({"prediction_time_utc": expected_hours}),
    how="cross",
)

static_features = [
    "elevation_mean_m",
    "terrain_endpoint_grade_abs",
    "terrain_relief_m",
    "length_m",
]
rain_features = [
    "rain_1h_mm", "rain_3h_mm", "rain_6h_mm", "rain_24h_mm"
]

coverage_rows = expected_rows.merge(
    static[
        ["street_segment_id", "grid_id", "cell_id",
         "static_features_valid"] + static_features
    ],
    on="street_segment_id",
    how="left",
    validate="many_to_one",
)

bridge_lookup = bridge[
    ["street_segment_id", "grid_id", "cell_id"]
].rename(columns={
    "grid_id": "model_grid_id",
    "cell_id": "model_cell_id",
})

coverage_rows = coverage_rows.merge(
    bridge_lookup,
    on="street_segment_id",
    how="left",
    validate="many_to_one",
)

coverage_rows["model_bridge_matches"] = (
    coverage_rows["grid_id"].eq(coverage_rows["model_grid_id"])
    & coverage_rows["cell_id"].eq(coverage_rows["model_cell_id"])
).fillna(False)

coverage_rows = coverage_rows.merge(
    weather[
        weather_keys + ["weather_features_valid"] + rain_features
    ],
    on=weather_keys,
    how="left",
    validate="many_to_one",
    indicator="weather_join",
)

coverage_rows["static_inputs_valid"] = (
    coverage_rows["static_features_valid"].eq(True)
    & np.isfinite(coverage_rows[static_features].to_numpy(
        dtype=float, na_value=np.nan
    )).all(axis=1)
)

rain_values = coverage_rows[rain_features].to_numpy(
    dtype=float, na_value=np.nan
)
coverage_rows["rainfall_inputs_valid"] = (
    coverage_rows["weather_features_valid"].eq(True)
    & np.isfinite(rain_values).all(axis=1)
    & (rain_values >= 0).all(axis=1)
    & (np.diff(rain_values, axis=1) >= -1e-6).all(axis=1)
)

coverage_rows["ready_for_inference"] = (
    coverage_rows["model_bridge_matches"]
    & coverage_rows["static_inputs_valid"]
    & coverage_rows["rainfall_inputs_valid"]
)

coverage_summary = coverage_rows.groupby(
    "lion_segment_id", as_index=False
).agg(
    expected_hours=("prediction_time_utc", "size"),
    model_supported_hours=("model_bridge_matches", "sum"),
    valid_static_hours=("static_inputs_valid", "sum"),
    valid_rainfall_hours=("rainfall_inputs_valid", "sum"),
    ready_hours=("ready_for_inference", "sum"),
)

coverage_summary["unavailable_hours"] = (
    coverage_summary["expected_hours"] - coverage_summary["ready_hours"]
)

display(coverage_summary)

unavailable_rows = coverage_rows.loc[
    ~coverage_rows["ready_for_inference"]
].copy()

if not unavailable_rows.empty:
    print("Unavailable rows requiring review:")
    display(unavailable_rows[[
        "lion_segment_id", "street_segment_id", "prediction_time_utc",
        "model_bridge_matches", "static_inputs_valid",
        "rainfall_inputs_valid", "weather_join",
    ]])

assert len(coverage_rows) == len(proposed_mapping) * len(expected_hours)

print(f"Expected segment-hours: {len(coverage_rows):,}")
print(f"Ready segment-hours: {coverage_rows['ready_for_inference'].sum():,}")
print(f"Unavailable segment-hours: {len(unavailable_rows):,}")
print("No missing feature values were filled with zero.")
print("No predictions generated; no files changed.")

,lion_segment_id,expected_hours,model_supported_hours,valid_static_hours,valid_rainfall_hours,ready_hours,unavailable_hours
0,21148,48,48,48,48,48,0
1,38916,48,48,48,48,48,0
2,81165,48,48,48,48,48,0
3,96010,48,48,48,48,48,0


Expected segment-hours: 192
Ready segment-hours: 192
Unavailable segment-hours: 0
No missing feature values were filled with zero.
No predictions generated; no files changed.


## 4. Actual Model Predictions

Run the existing climate release for all supported segment-hours.
Keep the model, history policies and alert threshold unchanged.

Retain all outputs, including zero alerts. Results remain an Ida
integration replay, not an independent evaluation.

In [4]:
import subprocess
import sys
import tempfile

base_features = release["native_external_features"]

inference_rows = coverage_rows.loc[
    coverage_rows["ready_for_inference"]
].sort_values(
    ["street_segment_id", "prediction_time_utc"]
).copy()

if inference_rows.duplicated(
    ["street_segment_id", "prediction_time_utc"]
).any():
    raise ValueError("Duplicate physical segment-hours.")

inference_rows["request_id"] = [
    f"ida-aligned:{row.street_segment_id}:"
    f"{row.prediction_time_utc:%Y%m%dT%H%M%SZ}"
    for row in inference_rows.itertuples(index=False)
]

native_request = {
    "format_version": request_spec["interface"],
    "package_id": release["release_id"],
    "records": [
        {
            "request_id": row.request_id,
            "street_segment_id": row.street_segment_id,
            "prediction_time_utc": (
                row.prediction_time_utc.isoformat().replace("+00:00", "Z")
            ),
            "features": {
                name: float(getattr(row, name))
                for name in base_features
            },
        }
        for row in inference_rows.itertuples(index=False)
    ],
}

# Preserve these exact bytes for the final export and its hashes.
request_bytes = (
    json.dumps(native_request, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

with tempfile.TemporaryDirectory(prefix="plexa-aligned-ida-") as temporary:
    temporary = Path(temporary)
    input_path = temporary / "request.json"
    output_path = temporary / "result.json"
    input_path.write_bytes(request_bytes)

    completed = subprocess.run(
        [
            sys.executable,
            str(BUNDLE_DIR / "predict.py"),
            "--bundle", str(BUNDLE_DIR),
            "--input", str(input_path),
            "--output", str(output_path),
        ],
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"Inference exited with code {completed.returncode}\n"
            f"{completed.stdout}\n{completed.stderr}"
        )

    result_bytes = output_path.read_bytes()

native_result = json.loads(result_bytes)

if native_result["request_sha256"] != hashlib.sha256(request_bytes).hexdigest():
    raise ValueError("Prediction results reference a different request.")

prediction_table = pd.json_normalize(native_result["records"])

if (
    len(prediction_table) != len(inference_rows)
    or prediction_table["request_id"].duplicated().any()
    or set(prediction_table["request_id"]) != set(inference_rows["request_id"])
    or not prediction_table["status"].eq("ok").all()
):
    raise ValueError("Prediction outputs do not reconcile with the request.")

if not prediction_table["result.model_alert_threshold"].eq(
    release["threshold"]
).all():
    raise ValueError("Unexpected alert threshold.")

prediction_table = prediction_table.merge(
    inference_rows[["request_id", "lion_segment_id"]],
    on="request_id",
    validate="one_to_one",
)

prediction_summary = prediction_table.groupby(
    "lion_segment_id", as_index=False
).agg(
    prediction_rows=("request_id", "size"),
    minimum_score=("result.reported_flooding_score", "min"),
    maximum_score=("result.reported_flooding_score", "max"),
    alert_hours=("result.model_alert", "sum"),
)

display(prediction_summary)

print(f"Actual predictions generated: {len(prediction_table):,}")
print("Frozen threshold:", release["threshold"])
print("Model package and policies unchanged.")
print("Exact request and result bytes retained in notebook memory.")
print("Final export packaging is next.")

,lion_segment_id,prediction_rows,minimum_score,maximum_score,alert_hours
0,21148,48,0.000003,0.000299,0
1,38916,48,0.000001,0.000190,0
2,81165,48,0.000001,0.000266,0
3,96010,48,0.000002,0.000318,0


Actual predictions generated: 192
Frozen threshold: 0.0014566994738777757
Model package and policies unchanged.
Exact request and result bytes retained in notebook memory.
Final export packaging is next.


In [5]:
EXPORT_DIR = (
    PROJECT_ROOT
    / "data/processed/climate_scenarios"
    / "ida-2021-lion-aligned-candidate-v1"
)

export_manifest = read_json(EXPORT_DIR / "candidate_manifest.json")

for name, expected_hash in export_manifest["files"].items():
    assert file_sha256(EXPORT_DIR / name) == expected_hash, name

assert (EXPORT_DIR / "request.json").read_bytes() == request_bytes
assert (EXPORT_DIR / "result.json").read_bytes() == result_bytes

print("Export hashes verified.")
print("Exported inputs and results exactly match this notebook.")
print("Package:", EXPORT_DIR)
print("Proposed mapping remains pending Anas's review.")

Export hashes verified.
Exported inputs and results exactly match this notebook.
Package: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_scenarios/ida-2021-lion-aligned-candidate-v1
Proposed mapping remains pending Anas's review.


In [6]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "config/climate_history_release_v3_02.json").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

REVIEW_DIR = (
    PROJECT_ROOT
    / "data/processed/climate_coverage_reviews/2024-2025-v1"
)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Verify the review files and the prepared assets they describe.
manifest = json.loads(
    (REVIEW_DIR / "review_manifest.json").read_text()
)
for name, expected in manifest["files"].items():
    if sha256_file(REVIEW_DIR / name) != expected:
        raise ValueError(f"Review file changed: {name}")

evidence = json.loads((REVIEW_DIR / "evidence.json").read_text())

for name, expected in evidence["asset_sha256"].items():
    if sha256_file(name) != expected:
        raise ValueError(f"Prepared asset changed: {name}")

monthly = pd.read_csv(REVIEW_DIR / "monthly_coverage.csv")
gaps = pd.read_csv(REVIEW_DIR / "affected_prediction_hours.csv")
road_coverage = pd.read_csv(
    REVIEW_DIR / "four_road_monthly_coverage.csv",
    dtype={"lion_segment_id": str},
)

display(Markdown("### Monthly rainfall coverage — UTC calendar months"))
display(
    monthly[[
        "month_utc",
        "expected_hours",
        "complete_grid_usable_hours",
        "incomplete_grid_hours",
        "valid_cell_percent",
        "absent_source_hours_including_lookback",
    ]].style.format({"valid_cell_percent": "{:.3f}%"})
)

display(Markdown("### Rainfall gaps"))

if gaps.empty:
    print("No unavailable prediction hours.")
else:
    gaps["prediction_time_new_york"] = (
        pd.to_datetime(gaps["prediction_time_utc"], utc=True)
        .dt.tz_convert("America/New_York")
    )

    display(
        gaps.groupby("month_utc", as_index=False).agg(
            first_unavailable_hour_NY=("prediction_time_new_york", "min"),
            last_unavailable_hour_NY=("prediction_time_new_york", "max"),
            unavailable_prediction_hours=("prediction_time_utc", "size"),
        )
    )

    print("First and last timestamps are inclusive.")
    print("Exact individual timestamps are retained in:", REVIEW_DIR / gaps.__class__.__name__
          if False else REVIEW_DIR / "affected_prediction_hours.csv")

print("Missing raw rainfall source hours — UTC:")
display(pd.DataFrame({
    "source_hour_utc": evidence["unique_missing_raw_source_hours"]
}))

display(Markdown("### Static features and native road support"))
display(pd.DataFrame({
    "Check": [
        "Native physical segments",
        "Segments with valid static features",
        "Matching saved-model bridge identities",
        "Prepared radar cells",
    ],
    "Count": [
        evidence["native_segments"],
        evidence["static_valid_segments"],
        evidence["model_bridge_matches"],
        evidence["prepared_radar_cells"],
    ],
}))

display(Markdown("### Previously supplied four LION roads"))
display(
    road_coverage.groupby("lion_segment_id", as_index=False).agg(
        months_checked=("month_utc", "nunique"),
        expected_hours=("expected_hours", "sum"),
        valid_input_hours=("valid_input_hours", "sum"),
        unavailable_input_hours=("unavailable_input_hours", "sum"),
    )
)

print("These four mappings remain proposed, pending Anas's review.")
print("Full mobility-registry coverage has not been established.")

display(Markdown("### Underlying prepared asset locations"))

bundle = (
    PROJECT_ROOT
    / "data/processed/climate_releases"
    / "climate-history-v3-02-experimental-01"
)

display(pd.DataFrame([
    {
        "Asset": "Monthly rainfall and per-month manifests",
        "Location": str(
            PROJECT_ROOT
            / "data/processed/baseline_weather_v1/YYYY-MM"
        ),
    },
    {
        "Asset": "Static street features",
        "Location": str(
            PROJECT_ROOT
            / "data/processed/baseline_street_features_v1/street_features.parquet"
        ),
    },
    {
        "Asset": "Native road geometry",
        "Location": str(
            PROJECT_ROOT
            / "data/processed/baseline_street_features_v1/street_segments.parquet"
        ),
    },
    {
        "Asset": "Physical-to-directed-edge bridge",
        "Location": str(
            PROJECT_ROOT / "data/processed/physical_street_index_v1.parquet"
        ),
    },
    {
        "Asset": "Model road/radar bridge",
        "Location": str(bundle / "street_cell_bridge.parquet"),
    },
    {
        "Asset": "Historical report events",
        "Location": str(bundle / "history_events.parquet"),
    },
    {
        "Asset": "Proposed LION crosswalk",
        "Location": str(
            PROJECT_ROOT
            / "data/processed/climate_scenarios"
            / "ida-2021-lion-aligned-candidate-v1/proposed_lion_crosswalk.csv"
        ),
    },
]))

print(
    "\nInterpretation: existing prepared inputs support historical playback. "
    "March 2024 gaps must remain explicit, not filled with zero."
)
print(
    "The model was trained into December 2024; a 2024-start replay "
    "is not an independent prospective test."
)
print(
    "UTC partition boundaries differ from New York calendar boundaries. "
    "Confirm exact dates and the road set before export."
)
print("Model, features, history policies and threshold unchanged.")
print("No predictions generated and no files changed.")

### Monthly rainfall coverage — UTC calendar months

,month_utc,expected_hours,complete_grid_usable_hours,incomplete_grid_hours,valid_cell_percent,absent_source_hours_including_lookback
0,2024-01,744,744,0,100.000%,0
1,2024-02,696,696,0,100.000%,0
2,2024-03,744,718,26,96.505%,3
3,2024-04,720,720,0,100.000%,0
4,2024-05,744,744,0,100.000%,0
5,2024-06,720,720,0,100.000%,0
6,2024-07,744,744,0,100.000%,0
7,2024-08,744,744,0,100.000%,0
8,2024-09,720,720,0,100.000%,0
9,2024-10,744,744,0,100.000%,0


### Rainfall gaps

,month_utc,first_unavailable_hour_NY,last_unavailable_hour_NY,unavailable_prediction_hours
0,2024-03,2024-03-19 16:00:00-04:00,2024-03-20 17:00:00-04:00,26


First and last timestamps are inclusive.
Exact individual timestamps are retained in: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_coverage_reviews/2024-2025-v1/affected_prediction_hours.csv
Missing raw rainfall source hours — UTC:


,source_hour_utc
0,2024-03-19T17:00:00+00:00
1,2024-03-19T18:00:00+00:00
2,2024-03-19T19:00:00+00:00


... (4 outputs left out)

These four mappings remain proposed, pending Anas's review.
Full mobility-registry coverage has not been established.


### Underlying prepared asset locations

,Asset,Location
0,Monthly rainfall and per-month manifests,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
1,Static street features,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
2,Native road geometry,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
3,Physical-to-directed-edge bridge,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
4,Model road/radar bridge,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
5,Historical report events,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...
6,Proposed LION crosswalk,/Users/macbookairm3/Documents/Plexa-A-Multi-Mo...



Interpretation: existing prepared inputs support historical playback. March 2024 gaps must remain explicit, not filled with zero.
The model was trained into December 2024; a 2024-start replay is not an independent prospective test.
UTC partition boundaries differ from New York calendar boundaries. Confirm exact dates and the road set before export.
Model, features, history policies and threshold unchanged.
No predictions generated and no files changed.
